# Documentação - Xadrez



**Tabuleiro**

Determinado por índices de 1 ... 64 em sistema linear

```
1  2  3  4  5  6  7  8
9 10 11 12 13 14 15 16
17 ...
...
57 58 59 60 61 62 63 64
```

Determinado por índices de (0, 0) ... (7, 7) em sistemas de coordenadas
```
(0, 0) (1, 0) (2, 0) (3, 0) (4, 0) (5, 0) (6, 0) (7, 0)
(0, 1) (1, 1) (2, 1) (3, 1) (4, 1) (5, 1) (6 ,1) (7, 1)
(0, 2) (1, 2) ...
...
(0, 7) (1, 7) (2, 7) (3, 7) (4, 7) (5, 7) (6, 7) (7, 7)
```

---

``` tabuleiro[y][x] = id + 1 ``` armazena o id da peça + 1 na casa (x, y), 0 se vazia

**Peças**

Cada peça possui um id que vai de 0 ... 15 para as pretas e 16 .. 31 para brancas

O valor ``` pecas[i] = pos ``` representa a posição linear da peça de id ``` i ```\
Se a peça foi capturada, sua posição é 0 (fora do tabuleiro)

Além disso, possuem um valor de **material** que representa o valor da peça

---

Conversão de posições lineares p/ coordenadas

```
x = (pieces[i]-1) % 8
y = (pecas[i]-1) // 8
```

Coluna: Fazemos um (mod 8) para iterar pelas colunas, já que cada linha possui 8 casas

Linha: Divisão inteira (```//```) por 8 pois dá exatamente o valor dde quantas linhas já passaram

Obs: ``` pieces[i]-1 ``` para converter a posição de índices que vão de 1 .. 64 para 0 ... 63. Fazemos isso para poder trabalhar com divisão por 8


# Imports

*abc* para utilizar classes abstratas

In [1]:
from abc import ABC, abstractmethod
import numpy as np
import time
import copy

# Movimento das Peças

In [2]:
def get_cor(id_):
  """
    False (0) se preta
    True (1) se branca

    parâmetros:
      id (int): id da peça
    retorna:
      cor (bool): cor da peça
  """
  return id_ > 15

def linear_para_coord(state, id_):
    """
      Transforma uma posição linear em coordenadas

      parâmetros:
        id (int): posição linear
      retorna:
        (x, y): posição em coordenadas
    """
    x = (state.pecas[id_]-1) % 8
    y = (state.pecas[id_]-1) // 8
    return (x, y)

def get_mat_add(state, id_):
  """
    Retorna o valor de captura de uma peça
  """
  return state.valor[id_]*(2*state.curr_player - 1)

In [3]:
def torre_casas_cobertas(x, y):
  """
    Gera todos as casas cobertas pela torre a partir de
      uma posição (x, y)

    par^amêtros:
      x (int): coluna
      y (int): linha
    retorna:
      mov (list): lista de movimentos
  """
  y_cima = range(y-1, -1, -1)
  cima = [y_cima, [x]]

  y_baixo = range(y+1, 8)
  baixo = [y_baixo, [x]]

  x_esq = range(x-1, -1, -1)
  esq = [[y], x_esq]

  x_dir = range(x+1, 8)
  dir_ = [[y], x_dir]

  return [cima, baixo, esq, dir_]

def torre_mov(state, movimentos):
  """
    Gera todos os possíveis movimentos para as torres do jogador atual.
    Consideramos a dama também pq ela anda que nem uma torre

    parâmetros:
      mov (ref lista): lista de movimentos, normalmente vazia e
        passada por referência
  """
  # Torres e Dama
  torres_pretas = [0, 7, 3]
  torres_brancas = [24, 31, 27]

  # Peças do jogador atual
  torres = [torres_pretas, torres_brancas][state.curr_player]

  for i in torres:
    if state.pecas[i] == 0:
      continue # Já capturada

    i_cor = get_cor(i)
    x, y = linear_para_coord(state, i)

    for range_y, range_x in torre_casas_cobertas(x, y):
      # Passamos por todas as possíveis casas da torre
      # range_y = range(...) e range_x = [x] ou
      # range_y = [y]     e    range_x = range(...)

      for casa_y in range_y:
        for casa_x in range_x:
          casa_peca = state.tabuleiro[casa_y][casa_x]
          if casa_peca == 0:
            # Casa vazia - Pode avançar
            movimentos.append([0, y, x, casa_y, casa_x, i+1, 0])

          elif i_cor + (casa_peca > 16) == 1:
            # Capturamos a peça
            mat_add = get_mat_add(state, casa_peca)
            movimentos.append([mat_add, y, x, casa_y, casa_x, i+1, casa_peca])
            break

          else:
            break

        else:
          continue

        break

In [4]:
def bispo_casas_cobertas(x, y):
  y_cima = range(y-1, -1, -1)
  y_baixo = range(y+1, 8)
  y_lista = [y_cima, y_baixo]

  x_esq = range(x-1, -1, -1)
  x_dir = range(x+1, 8)
  x_lista = [x_esq, x_dir]

  return [y_lista, x_lista]


def bispo_mov(state, movimentos):
  # Consideramos a dama como bispo
  bispos_pretos = [2, 5, 3]
  bispos_brancos = [26, 29, 27]

  bispos = [bispos_pretos, bispos_brancos][state.curr_player]

  for i in bispos:
    if state.pecas[i] == 0:
      continue

    i_cor = get_cor(i)
    x, y = linear_para_coord(state, i)

    casas_cobertas = bispo_casas_cobertas(x, y)

    for range_y in casas_cobertas[0]:
      for range_x in casas_cobertas[1]:
        eixo_minimo = min(len(range_y), len(range_x))
        passos = range(eixo_minimo)
        for p in passos:
          casa_y = range_y[p]
          casa_x = range_x[p]

          casa_peca = state.tabuleiro[casa_y][casa_x]

          if casa_peca == 0:
            movimentos.append([0, y, x, casa_y, casa_x, i+1, 0])

          elif i_cor + (casa_peca > 16) == 1:
            mat_add = get_mat_add(state, casa_peca)
            movimentos.append([mat_add, y, x, casa_y, casa_x, i+1, casa_peca])
            break

          else:
            break

In [5]:
def cavalo_mov(state, movimentos):
  cavalos_pretos = [1, 6]
  cavalos_brancos = [25, 30]

  cavalos = [cavalos_pretos, cavalos_brancos][state.curr_player]

  for i in cavalos:
    if state.pecas[i] == 0:
      continue

    i_cor = get_cor(i)
    x, y = linear_para_coord(state, i)

    for dist_y in [1, 2]:
      for dir_y in [-1, 1]:
        casa_y = y + dist_y * dir_y

        if casa_y < 0 or casa_y > 7:
          continue

        for dir_x in [-1, 1]:
          casa_x = x + (3 - dist_y) * dir_x

          if casa_x < 0 or casa_x > 7:
            continue

          casa_peca = state.tabuleiro[casa_y][casa_x]

          if casa_peca == 0 or ((i_cor) + (casa_peca > 16) == 1):
            mat_add = get_mat_add(state, casa_peca)
            movimentos.append([mat_add, y, x, casa_y, casa_x, i+1, casa_peca])

In [15]:
def peao_mov(state, movimentos):
  dir_ = 1 - state.curr_player*2

  peoes_pretos = range(8, 16)
  peoes_brancos = range(16, 24)

  peoes = [peoes_pretos, peoes_brancos][state.curr_player]

  for i in peoes:
    if state.pecas[i] == 0:
      continue

    i_cor = get_cor(i)
    x, y = linear_para_coord(state, i)

    if y == 0 or y == 7:
      continue

    if x > 0 and state.tabuleiro[y+dir_][x-1] != 0 and (i_cor + (state.tabuleiro[y+dir_][x-1] > 16)) == 1:
      mat_add = get_mat_add(state, state.tabuleiro[y+dir_][x-1])
      movimentos.append([mat_add, y, x, y+dir_, x-1, i+1, state.tabuleiro[y+dir_][x-1]])

    if x < 7 and state.tabuleiro[y+dir_][x+1] != 0 and (i_cor + (state.tabuleiro[y+dir_][x+1] > 16)) == 1:
      mat_add = get_mat_add(state, state.tabuleiro[y+dir_][x+1])
      movimentos.append([mat_add, y, x, y+dir_, x+1, i+1, state.tabuleiro[y+dir_][x+1]])

    if state.tabuleiro[y+dir_][x] == 0:
      movimentos.append([0, y, x, y+dir_, x, i+1, 0])

      if y == [1, 6][state.curr_player] and state.tabuleiro[y + dir_*2][x] == 0:
        movimentos.append([0, y, x, y+dir_*2, x, i+1, 0])

In [7]:
def rei_casas_cobertas(x, y):
  cima = max(0, y-1)
  baixo = min(8, y+2)
  y_lista = range(cima, baixo)

  esq = max(0, x-1)
  dir_ = min(8, x+2)
  x_lista = range(esq, dir_)


  return [y_lista, x_lista]

def rei_mov(state, movimentos):
  rei = [4, 28][state.curr_player]

  i_cor = get_cor(rei)
  x, y = linear_para_coord(state, rei)

  casas_cobertas = rei_casas_cobertas(x, y)

  for casa_y in casas_cobertas[0]:
    for casa_x in casas_cobertas[1]:
      casa_peca = state.tabuleiro[casa_y][casa_x]
      if casa_peca == 0 or (i_cor + (casa_peca > 16) == 1):
        mat_add = get_mat_add(state, casa_peca)
        movimentos.append([mat_add, y, x, casa_y, casa_x, rei+1, casa_peca])

# Classes


In [8]:
class Game(ABC):
  def __init__(self):
    pass

  @abstractmethod
  def get_next_state(self, state, action, player):
    pass

  @abstractmethod
  def get_valid_moves(self, state):
    pass

  @abstractmethod
  def get_value_and_terminated(self, value):
    pass

  @abstractmethod
  def get_opponent(self, player):
    pass

  @abstractmethod
  def get_opponent_value(self, value):
    pass

In [9]:
class Xadrez(Game):
  def __init__(self):
    self.initial_state()

  def initial_state(self):
    """
      Inicia o Estado inicial do jogo
    """
    self.tabuleiro = [[i for i in range(1,9)],
      [i for i in range(9,17)],
      [0]*8,
      [0]*8,
      [0]*8,
      [0]*8,
      [i for i in range(17,25)],
      [i for i in range(25,33)]]

    self.pecas = [*[i for i in range(1,17)], *[i for i in range(49,65)]]
    self.posicoes = {tuple(self.pecas):1}

    self.valor = [0, 5, 3, 3, 9, float("inf"), 3, 3, 5,
              1, 1, 1, 1, 1, 1, 1, 1,
              1, 1, 1, 1, 1, 1, 1, 1,
              5, 3, 3, 9, float("inf"), 3, 3, 5]

    self.valor_casa = [[0.50,0.52,0.55,0.57,0.57,0.55,0.52,0.50],
                  [0.52,0.55,0.60,0.70,0.70,0.60,0.55,0.52],
                  [0.55,0.60,0.70,0.80,0.80,0.70,0.60,0.55],
                  [0.57,0.65,0.85,1.00,1.00,0.85,0.65,0.57],
                  [0.57,0.65,0.85,1.00,1.00,0.85,0.65,0.57],
                  [0.55,0.60,0.70,0.80,0.80,0.70,0.60,0.55],
                  [0.52,0.55,0.60,0.70,0.70,0.60,0.55,0.52],
                  [0.50,0.52,0.55,0.57,0.57,0.55,0.52,0.50]]

    self.string = {0:"OO", 1:"BR", 2:"BN", 3:"BB", 4:"BQ", 5:"BK", 6:"BB", 7:"BN", 8:"BR",
                    9:"BP", 10:"BP", 11:"BP", 12:"BP", 13:"BP", 14:"BP", 15:"BP", 16:"BP",
                    17:"WP", 18:"WP", 19:"WP", 20:"WP", 21:"WP", 22:"WP", 23:"WP", 24:"WP",
                    25:"WR", 26:"WN", 27:"WB", 28:"WQ", 29:"WK", 30:"WB", 31:"WN", 32:"WR"}

    self.curr_player = 1

    self.prof = 0
    self.max_prof = 5
    self.material = 0

  def do_action(self, action, verbose):
    """
      Anda o próprio jogo em após a
        ação do player atual

      parâmetros:
        action: Ação a ser executada em texto. Ex: "e2e4"
        player: Jogador atual
      retorno:
        True se o jogo ainda continua
        False se o jogo acabou
    """
    time_ = time.time()

    # Traduz a ação de string para index
    from_x = "abcdefgh".index(action[0])
    from_y = 8-int(action[1])
    to_x = "abcdefgh".index(action[2])
    to_y = 8-int(action[3])

    # Descobre as pecas
    peca = self.tabuleiro[from_y][from_x]
    capturada = self.tabuleiro[to_y][to_x]
    mat_add = -self.valor[capturada]

    # Atualiza o tabuleiro
    self.tabuleiro[from_y][from_x] = 0
    self.tabuleiro[to_y][to_x] = peca
    self.material += mat_add
    self.curr_player = 1 - self.curr_player

    # Atualiza a lista de peças
    self.pecas[peca-1] = to_y * 8 + to_x + 1
    if capturada != 0:
        self.pecas[capturada-1] = 0

    # Checa empate
    if tuple(self.pecas) in self.posicoes:
        self.posicoes[tuple(self.pecas)] += 1
        if self.posicoes[tuple(self.pecas)] == 3:
            print("Tie")
            return False
    else:
        self.posicoes[tuple(self.pecas)] = 1

    # Imprime se pedido
    if verbose:
      print(time.time() - time_)
      for x in self.tabuleiro:
          line = ""
          for peca in x:
              line += self.string[peca] + " "
          print(line)
      print("")

    # Verifica fim de jogo
    # WARNING: o valor de infinito é setado especificamente
    #   para o minmax, pode quebrar se usar outro
    if self.material == float("inf"):
        print("Brancas vencem!")
        return False
    elif self.material == -float("inf"):
        print("Pretas vencem!")
        return False

    return True

  def get_next_state(self, state, action, player=-1, verbose=False):
    """
      Aplica a ação do player ao estado e retorna um novo estado.

      parâmetros:
        state: Estado atual
        action: Ação a ser executada
        player: Player a realizar a ação
      retorno:
        Novo estado com a ação executada
    """
    new_state = copy.deepcopy(state)

    if player != -1: new_state.player = player

    new_state.do_action(action, player, verbose)
    return new_state

  def get_valid_moves(self, state):
    """
      Retorna as ações possíveis para o estado atual.

      parâmetros:
        state: Estado atual
      retorno:
        Lista com todos os estados possíveis
    """
    moves = []
    torre_mov(self, moves)
    bispo_mov(self, moves)
    cavalo_mov(self, moves)
    peao_mov(self, moves)
    rei_mov(self, moves)
    return moves

  def get_value_and_terminated(self, state, action):
    """
    Retorna:
      value: resultado do ponto de vista de quem jogou a action
      is_terminal: se o jogo terminou
    """

    # Pretas ganharam
    if state.material == -float("inf"):
        return -1, True

    # Brancas ganharam
    if state.material == float("inf"):
        return 1, True

    # Empate
    if tuple(state.pecas) in state.posicoes and state.posicoes[tuple(state.pecas)] >= 3:
        return 0, True

    # Jogo ainda não acabou
    return 0, False

  def get_opponent(self, player):
    return -player

  def get_opponent_value(self, value):
    return -value

# Jogo manual

Utilizando minimax

## Funções de busca e avaliação (minimax)

### Função de avaliação

In [10]:
def avaliacao(state):
  global  valor, curr_player

  controle = 0

  movimentos = []
  torre_mov(state, movimentos)
  bispo_mov(state, movimentos)
  cavalo_mov(state, movimentos)
  peao_mov(state, movimentos)
  rei_mov(state, movimentos)

  k = 0.5

  for mat_add, from_y, from_x, to_y, to_x, peca, capturada in movimentos:
    controle += state.valor_casa[to_y][to_x] ** (1/3)

    if abs(mat_add) > 1:
      if state.valor[peca] < abs(mat_add):
        controle += min(9,abs(mat_add))*2*k
      elif state.valor[peca] == abs(mat_add):
        controle += min(9,abs(mat_add))*1.5*k
      else:
        controle += min(9,abs(mat_add))*k

  controle_oponente = 0

  state.curr_player = 1 - state.curr_player

  movimentos = []
  torre_mov(state, movimentos)
  bispo_mov(state, movimentos)
  cavalo_mov(state, movimentos)
  peao_mov(state, movimentos)
  rei_mov(state, movimentos)

  for mat_add, from_y, from_x, to_y, to_x, peca, capturada in movimentos:
    controle_oponente += state.valor_casa[to_y][to_x] ** (1/3)

    if abs(mat_add) > 1:
      if state.valor[peca] < abs(mat_add):
        controle_oponente += min(9,abs(mat_add))*2*k
      elif state.valor[peca] == abs(mat_add):
        controle_oponente += min(9,abs(mat_add))*1.5*k
      else:
        controle_oponente += min(9,abs(mat_add))*k


  state.curr_player = 1 - state.curr_player

  return state.material + controle*(2*state.curr_player - 1)/20 - controle_oponente*(2*state.curr_player - 1)/20

### Algoritmo de busca

In [11]:
def minimax(state, alpha = float("inf"), beta = -float("inf")):
  # repetição
  if state.prof > 0 and tuple(state.pecas) in state.posicoes:
    return 0

  # avaliacao - termina busca
  if state.prof == state.max_prof:
    return avaliacao(state)

  # encontra movimentos
  movimentos = []
  torre_mov(state, movimentos)
  bispo_mov(state, movimentos)
  cavalo_mov(state, movimentos)
  peao_mov(state, movimentos)
  rei_mov(state, movimentos)

  # ordenando por relevância
  if state.curr_player == 1:
    movimentos.sort(reverse=True, key=lambda x:x[0]+len(x)/100)
  else:
    movimentos.sort(key=lambda x:x[0]-len(x)/100)

  melhor = float("inf")*(1 - 2*state.curr_player)
  jogada = []

  state.prof += 1

  # buscando mais a fundo
  for mat_add, from_y, from_x, to_y, to_x, peca, capturada in movimentos:
    #xeque-mate encontrado
    if abs(mat_add) == float("inf"):
      melhor = mat_add
      movimento = [mat_add, from_y, from_x, to_y, to_x, peca, capturada]

      if state.curr_player == 1:
        beta = max(beta, mat_add)
      else:
        alpha = min(alpha, mat_add)

    if beta >= alpha:
      break

    # fazendo jogada
    state.tabuleiro[from_y][from_x] = 0
    state.tabuleiro[to_y][to_x] = peca

    state.pecas[peca-1] = to_y * 8 + to_x + 1
    if capturada != 0:
      state.pecas[capturada-1] = 0

    state.material += mat_add

    state.curr_player = 1 - state.curr_player

    aval_mov = minimax(state, alpha, beta)

    state.curr_player = 1 - state.curr_player

    if state.curr_player == 1:
      if aval_mov >= melhor:
        beta = max(beta, aval_mov)
        melhor = aval_mov
        movimento = [mat_add, from_y, from_x, to_y, to_x, peca, capturada]
    else:
      if aval_mov <= melhor:
        alpha = min(alpha, aval_mov)
        melhor = aval_mov
        movimento = [mat_add, from_y, from_x, to_y, to_x, peca, capturada]

    # desfazendo movimento
    state.tabuleiro[from_y][from_x] = peca
    state.tabuleiro[to_y][to_x] = capturada

    state.pecas[peca-1] = from_y * 8 + from_x + 1
    if capturada != 0:
      state.pecas[capturada-1] = to_y * 8 + to_x + 1

    state.material -= mat_add

  state.prof -= 1

  if state.prof != 0:
    return melhor

  return [*movimento, melhor]

## Loop de jogo

In [23]:
x = Xadrez()
x_axis = "abcdefgh"


while True:
  mat_add, from_y, from_x, to_y, to_x, peca, capturada, melhor = minimax(x)
  print(from_y, to_y)
  from_x = x_axis[from_x]
  to_x = x_axis[to_x]
  from_y = 8-from_y
  to_y = 8-to_y
  print(from_y, to_y)

  action = from_x + str(from_y) + to_x + str(to_y)

  print(action)

  continua = x.do_action(action, True)

  if not continua:
    break

  input_ = input().strip()
  continua = x.do_action(input_, True)

  if not continua:
    break

6 5
2 3
e2e3
1.6927719116210938e-05
BR BN BB BQ BK BB BN BR 
BP BP BP BP BP BP BP BP 
OO OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO WP OO OO OO 
WP WP WP WP OO WP WP WP 
WR WN WB WQ WK WB WN WR 

a7a6
1.430511474609375e-05
BR BN BB BQ BK BB BN BR 
OO BP BP BP BP BP BP BP 
BP OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO WP OO OO OO 
WP WP WP WP OO WP WP WP 
WR WN WB WQ WK WB WN WR 

7 4
1 4
d1g4
1.7881393432617188e-05
BR BN BB BQ BK BB BN BR 
OO BP BP BP BP BP BP BP 
BP OO OO OO OO OO OO OO 
OO OO OO OO OO OO OO OO 
OO OO OO OO OO OO WQ OO 
OO OO OO OO WP OO OO OO 
WP WP WP WP OO WP WP WP 
WR WN WB OO WK WB WN WR 



KeyboardInterrupt: Interrupted by user

# AlphaZero

fonte: https://youtu.be/wuSQpLinRB4?si=ymmOG1WCWwqh5J6j

https://www.youtube.com/watch?v=gsbkPpoxGQk

Alphazero usa uma rede neural de função: f0(s) -> (p,v)

*   s = posição
*   p = policy é a distribuição das probabilidades de cada ação, ele da a probabilidade de cada lance possível ser mais promissor(auxilia o MCTS a explorar os caminhos mais promissores)
*   v = value varia de -1 a 1 e representa uma nota para cada posição, que é a estimativa do resultado final a partir da posição s (evita a fase de jogar aleatoriamente até o fim no MCTS)
*   a rede primeiro chuta policy(as probabilidades de cada lance) e o algoritmo MCTS faz as expansões/simulações e contabiliza n(quantas vezes MCTS visitou cada lance a partir da posição atual) para gerar π: “probabilidades segundo a busca do MCTS",
*   π vira o gabarito para treinar a policy da rede, e o resultado final do jogo (z, vitória/empate/derrota) vira o target para treinar o value.


## Monte Carlo Search Tree (MCTS)

In [ ]:
class Node:
  def __init__(self, game, args, state, parent=None, action_taken=None):
    """
      Propriedades:
        game: Classe do jogo
        args: Parâmetros da árvore
        state: Estado do jogo
        parent: Pai
        action_taken: Ação tomada para chegar no estado atual

        children: Filhos
        expandable_moves: Possíveis ações que podem ser executadas
          a partir do estado

        visit_count: Número de vezes que o nó foi visitado (a partir das folhas)
        value_sum: Soma dos valores dos nós visitados
    """
    self.game = game
    self.args = args
    self.state = state
    self.parent = parent
    self.action_taken = action_taken

    self.children = []
    self.expandable_moves = game.get_valid_moves(state)

    self.visit_count = 0
    self.value_sum = 0

  def is_fully_expanded(self):
    """
      Retorna True se todas as ações possíveis já foram exploradas.
    """
    return np.sum(self.expandable_moves) == 0 and len(self.children) > 0

  def select(self):
    """
      Escolhe o filho com maior valor UCB.
    """
    best_child = None
    best_ucb = -np.inf

    for child in self.children:
      ucb = self.get_ucb(child)
      if ucb > best_ucb:
        best_child = child
        best_ucb = ucb

    return best_child

  def simulate(self):
    """
      Simula o jogo a partir do estado atual e cria novos nós para cada
        nova jogada.
    """
    value, is_terminal = self.game.get_value_and_terminated(self.state, self.action_taken)
    value = self.game.get_opponent_value(value)

    if is_terminal:
        return value

    curr_state = self.state.copy()
    future_state = None
    future_player = 1

    # Vamos simular algumas ações selecionadas aleatoriamente (Monte-Carlo)
    # Até o final do jogo
    while True:
        valid_moves = self.game.get_valid_moves(curr_state)
        action = np.random.choice(np.where(valid_moves == 1)[0])
        future_state = self.game.get_next_state(curr_state, action, future_player)
        value, is_terminal = self.game.get_value_and_terminated(future_state, action)
        if is_terminal:
            if future_player == -1:
                value = self.game.get_opponent_value(value)
            return value

        future_player = self.game.get_opponent(future_player)

  def get_ucb(self, child):
    """
      Retorna o valor UCB de um filho.

      UCB = Q(s,a) + pi * C * sqrt(ln(N) / n)

      Q pode ser interpretado como uma métrica de WinRate
    """
    win_rate = ((child.value_sum / child.visit_count) + 1) /2
    return win_rate + self.args['C'] * np.sqrt(np.log(self.visit_count) / child.visit_count)

  def backpropagate(self, value):
    """
      Realiza o backpropagation do valor do nó até a raiz, atualizando o valor
        ucb e o número de visitas de cada nó.
    """
    self.value_sum += value
    self.visit_count += 1

    value = self.game.get_opponent_value(value)
    if self.parent is not None:
        self.parent.backpropagate(value)


class MCTS:
    def __init__(self, game, args):
        self.game = game
        self.args = args

    def search(self, state):
        root = Node(self.game, self.args, state)

        for search in range(self.args['num_searches']):
            node = root

            while node.is_fully_expanded():
                node = node.select()
            #selection
            #expansion
            #simulation
            #backpropagation

          #return visit_counts
            value, is_terminal = self.game.get_value_and_terminated(node.state, node.action_taken)
            value = self.game.get_opponent_value(value)

            if not is_terminal:
                node = node.expand()
                value = node.simulate()

            node.backpropagate(value)


        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        return action_probs
